# Endogenous Reachability Collapse Model (ERCM)

**A Minimal Dynamical System for Loss of Recoverability Without Loss of State Space**

This notebook reproduces the main numerical results used in the ERCM preprint. It is aligned with the manuscript text and retains the **Gaussian forcing term** used in the working model that generated the reported collapse transition.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

## Model parameters

We use:
- two fast variables: $x(t), y(t)$
- one slow internal variable: $z(t)$
- a thresholded memory term in $z$
- a Gaussian forcing term $u(t)$, retained because it is part of the parameter regime that produced the reported transition in reachable fraction


In [ ]:
# Core model parameters
delta = 0.6
epsilon = 0.05
beta = 0.3
gamma = 0.25
x_c = 0.5

# Gaussian forcing parameters
A = 1.0
t0 = 10.0
sigma = 2.0

# Simulation controls
Tmax = 80.0
npts = 3000
t_eval = np.linspace(0, Tmax, npts)

# Recoverability definition
x_rec_candidates = [-1.67, 0.85]
rx = 0.25
ry = 0.25
final_window = 10.0

print("Parameters loaded.")

## Dynamical system

The system is:

\[
\dot{x} = y
\]

\[
\dot{y} = -\delta y + (x - x^3) - \alpha z\,\max(0,x) + u(t)
\]

\[
\dot{z} = \varepsilon(\beta x^2 - z) + \gamma\,\max(0, x - x_c)
\]

with Gaussian forcing

\[
u(t) = A\exp\left[-\frac{1}{2}\left(\frac{t-t_0}{\sigma}\right)^2\right].
\]


In [ ]:
def forcing(t, A=A, t0=t0, sigma=sigma):
    return A * np.exp(-0.5 * ((t - t0) / sigma) ** 2)

def rhs(t, s, alpha):
    x, y, z = s
    u = forcing(t)

    dx = y
    dy = -delta * y + (x - x**3) - alpha * z * np.maximum(0.0, x) + u
    dz = epsilon * (beta * x**2 - z) + gamma * np.maximum(0.0, x - x_c)

    return [dx, dy, dz]

def simulate(x0, y0, z0=0.0, alpha_value=1.0):
    sol = solve_ivp(
        lambda t, s: rhs(t, s, alpha_value),
        [0.0, Tmax],
        [x0, y0, z0],
        t_eval=t_eval,
        rtol=1e-7,
        atol=1e-9,
    )
    return sol.t, sol.y

def did_return(t, Y, x_rec_list=x_rec_candidates, rx=rx, ry=ry, final_window=final_window):
    x, y, z = Y
    mask = t >= (t.max() - final_window)

    for x_rec in x_rec_list:
        in_region = (
            (np.abs(x[mask] - x_rec) < rx) &
            (np.abs(y[mask]) < ry)
        )
        if np.any(in_region):
            return True
    return False

## Example trajectory

This representative trajectory is not the main result; it is a sanity check showing the model evolves in the intended regime.


In [ ]:
alpha_example = 1.0
t, Y = simulate(0.2, 0.0, 0.0, alpha_value=alpha_example)
x, y, z = Y

returned = did_return(t, Y)

print("Returned to any recovered region?", returned)
print("Final x:", x[-1])
print("Final y:", y[-1])
print("Final z:", z[-1])

plt.figure(figsize=(9, 4))
plt.plot(t, x, label='x')
plt.plot(t, z, label='z')
for xc in x_rec_candidates:
    plt.axhline(xc, linestyle='--', linewidth=1)
plt.xlabel("time")
plt.ylabel("state")
plt.title(f"Example trajectory (alpha={alpha_example:.2f})")
plt.legend()
plt.grid(True)
plt.show()

## Reachability map

For each initial condition $(x_0,y_0)$, we test whether the trajectory returns to the recovered regime.


In [ ]:
def recovery_map(x_vals, y_vals, z0=0.0, alpha_value=1.0):
    R = np.zeros((len(y_vals), len(x_vals)))

    for i, y0 in enumerate(y_vals):
        for j, x0 in enumerate(x_vals):
            t, Y = simulate(x0=x0, y0=y0, z0=z0, alpha_value=alpha_value)
            R[i, j] = 1.0 if did_return(t, Y) else 0.0

    return R

In [ ]:
x_vals = np.linspace(-1.8, 1.8, 21)
y_vals = np.linspace(-1.8, 1.8, 21)

R_low = recovery_map(x_vals, y_vals, z0=0.0, alpha_value=0.8)
R_high = recovery_map(x_vals, y_vals, z0=0.0, alpha_value=1.2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(
    R_low,
    extent=[x_vals.min(), x_vals.max(), y_vals.min(), y_vals.max()],
    origin='lower',
    aspect='auto',
    vmin=0,
    vmax=1
)
axes[0].set_title('LOW constraint')
axes[0].set_xlabel('x0')
axes[0].set_ylabel('y0')

im1 = axes[1].imshow(
    R_high,
    extent=[x_vals.min(), x_vals.max(), y_vals.min(), y_vals.max()],
    origin='lower',
    aspect='auto',
    vmin=0,
    vmax=1
)
axes[1].set_title('HIGH constraint')
axes[1].set_xlabel('x0')
axes[1].set_ylabel('y0')

fig.colorbar(im1, ax=axes.ravel().tolist(), label='return (1=yes, 0=no)')
plt.suptitle('Recovery basin under low and high internal constraint')
plt.tight_layout()
plt.show()

## Coarse sweep

We compute the fraction of initial conditions that return to the recovered regime as $\alpha$ increases.


In [ ]:
alpha_values = np.linspace(0.5, 3.0, 8)
fractions = []

for a in alpha_values:
    R = recovery_map(x_vals, y_vals, z0=0.0, alpha_value=a)
    frac = np.mean(R)
    fractions.append(frac)
    print(f"alpha={a:.3f}, reachable fraction={frac:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(alpha_values, fractions, '-o')
plt.xlabel('alpha (internal constraint)')
plt.ylabel('reachable fraction')
plt.title('Reachable set collapse')
plt.grid(True)
plt.show()

## Refined sweep near the transition

A faster grid is used here to resolve the transition more efficiently.


In [ ]:
x_vals_fast = np.linspace(-1.8, 1.8, 21)
y_vals_fast = np.linspace(-1.8, 1.8, 21)

alpha_values_fine = np.linspace(0.85, 1.20, 15)
fractions_fine = []

for a in alpha_values_fine:
    R = recovery_map(x_vals_fast, y_vals_fast, z0=0.0, alpha_value=a)
    frac = np.mean(R)
    fractions_fine.append(frac)
    print(f"alpha={a:.3f}, reachable fraction={frac:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(alpha_values_fine, fractions_fine, '-o')
plt.xlabel('alpha (internal constraint)')
plt.ylabel('reachable fraction')
plt.title('Refined transition in reachable set size')
plt.grid(True)
plt.show()

## Final figure

This is the main figure used to support the manuscript claim.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(alpha_values, fractions, '-o', label='coarse sweep')
plt.plot(alpha_values_fine, fractions_fine, '-s', label='refined sweep')
plt.axvline(1.05, linestyle='--', color='black', label='critical α*')

plt.xlabel('Internal constraint strength (alpha)')
plt.ylabel('Reachable fraction')
plt.title('Reachable set collapse under increasing internal constraint')
plt.ylim(-0.02, 1.02)
plt.grid(True)
plt.legend()

plt.savefig('figure_reachability_collapse.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: figure_reachability_collapse.png")

## Notes

- This notebook is aligned with the manuscript text claiming high recoverability below a critical range and collapse above it.
- The current result is a **minimal constructive example**.
- It does **not** establish topological trajectory exclusion; it supports endogenous collapse of recoverability/accessibility.
